# Camargo Helpdesk — SharedCat architecture + role-grouped pickles

Sibling of `train_camargo_LSTM.ipynb`. Two changes relative to the original training run:

1. Loads the role-grouped pickles produced by `loader_notebooks/camargo/Helpdesk_full_loader_with_roles.ipynb` (`helpdesk_all_5_roles_*`.pkl`). `model_feat` becomes `[['Activity', 'Role'], ['case_elapsed_time']]`.
2. Uses `sharedCatLSTM.model.SharedCat_LSTM` instead of `FullShared_Join_LSTM` — the paper's Fig 6b variant where the shared LSTM only sees categorical embeddings and numeric features are injected into the specialised head LSTM. Table 3 of the paper shows this architecture is the best performer on Helpdesk (DL act sim 0.9568 vs 0.5773 for the full-shared variant).

All training hyper-parameters match the original run so that the delta to the reported 0.789 next-event accuracy can be attributed cleanly to the architecture + pre-processing changes.

# Imports

In [ ]:
import importlib
import sys
import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')

# Data (role-grouped pickles)

In [ ]:
file_path_train = '../../../../../../encoded_data/compare_camargo/helpdesk_all_5_roles_train.pkl'
helpdesk_train_dataset = torch.load(file_path_train, weights_only=False)
print(type(helpdesk_train_dataset))

file_path_val = '../../../../../../encoded_data/compare_camargo/helpdesk_all_5_roles_val.pkl'
helpdesk_val_dataset = torch.load(file_path_val, weights_only=False)
print(type(helpdesk_val_dataset))

In [ ]:
helpdesk_all_categories = helpdesk_train_dataset.all_categories
helpdesk_all_categories_cat = helpdesk_all_categories[0]
helpdesk_all_categories_num = helpdesk_all_categories[1]
print(helpdesk_all_categories_cat)
print(helpdesk_all_categories_num)

for i, cat in enumerate(helpdesk_all_categories_cat):
    print(f'Categorical feature: {cat[0]}, idx {i}, #classes={cat[1]}')
for i, num in enumerate(helpdesk_all_categories_num):
    print(f'Numerical feature:   {num[0]}, idx {i}')

concept_name = 'Activity'
concept_name_id = [i for i, cat in enumerate(helpdesk_all_categories_cat) if cat[0] == concept_name][0]
concept_name_size = [cat[1] for cat in helpdesk_all_categories_cat if cat[0] == concept_name][0]
eos_id = [v for k, v in helpdesk_all_categories_cat[concept_name_id][2].items() if k == 'EOS'][0]
print('Activity idx / size / EOS id:', concept_name_id, concept_name_size, eos_id)

In [ ]:
model_feat_cat = [cat[0] for cat in helpdesk_all_categories_cat]
model_feat_num = [num[0] for num in helpdesk_all_categories_num]
model_feat = [model_feat_cat, model_feat_num]
print('model_feat:', model_feat)
assert 'Role' in model_feat_cat, 'Expected the role-grouped pickle. Regenerate it with Helpdesk_full_loader_with_roles.ipynb.'

# Model (SharedCat)

In [ ]:
import sharedCatLSTM.model
importlib.reload(sharedCatLSTM.model)
from sharedCatLSTM.model import SharedCat_LSTM

# Same hyper-parameters as the original paper / our reproduction
hidden_size = 50
num_layers = 1
input_size = 1  # sentinel -> SharedCat_LSTM will compute it from embeddings

model = SharedCat_LSTM(
    data_set_categories=helpdesk_all_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    model_feat=model_feat,
    input_size=input_size,
    output_size_act=concept_name_size,
)

# Training

In [ ]:
import training.train
importlib.reload(training.train)
from training.train import Training

from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(comment='Full_helpdesk_camargo_sharedcat_roles')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

learning_rate = 1e-5
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=0)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4, min_lr=1e-10)

num_epochs = 100
batch_size = 128
shuffle = True

optimize_values = {
    'optimizer': optimizer,
    'scheduler': scheduler,
    'epochs': num_epochs,
    'mini_batches': batch_size,
    'shuffle': shuffle,
}

trainer = Training(
    model=model,
    device=device,
    data_train=helpdesk_train_dataset,
    data_val=helpdesk_val_dataset,
    optimize_values=optimize_values,
    concept_name_id=concept_name_id,
    eos_id=eos_id,
    writer=writer,
    save_model_n_th_epoch=1,
    saving_path='Helpdesk_camargo_sharedcat_roles_act_1_suffix_length5.pkl',
)

trainer.train()